In [0]:
from pyspark.sql.functions import col, coalesce, when, to_date, lit
from delta.tables import DeltaTable

bronze_df = spark.read.table("payment_gateway_catalog.bronze.bronze_payment_raw")

silver_transformed = bronze_df.select(
    coalesce(col("data.payment_id"), col("data.transaction_id")).alias("transaction_id"),
    col("provider"),
    col("timestamp").cast("timestamp").alias("event_timestamp"),
    (col("data.amount_subunits")/100.0).cast("decimal(12,2)").alias("original_amount"),
    col("data.currency").alias("original_currency"),
    coalesce(col("data.fx_rate_to_inr"), lit(1.0)).cast("decimal(8,4)").alias("fx_rate"),
    col("data.status").alias("status"),
    when(col("data.status").isin("captured", "succeeded"), True).otherwise(False).alias("is_success"),
    col("data.error.code").alias("error_code"),
    col("data.error.description").alias("error_description"),
    col("_ingested_at").alias("processed_at"),

    # Razorpay fields
    col("data.method").alias("method"),
    col("data.vpa").alias("vpa"),
    col("data.fee").alias("fee"),
    col("data.tax").alias("tax"),

    # Slice fields
    col("data.tenure_months").alias("tenure_months"),
    col("data.merchant_category_code").alias("merchant_category_code"),

    # Dodo Payments fields
    col("data.tax_jurisdiction").alias("tax_jurisdiction"),
    col("data.vat_or_sales_tax_collected").alias("vat_or_sales_tax_collected")
).withColumn(
    "amount_inr", (col("original_amount") * col("fx_rate")).cast("decimal(12,2)")).withColumn("event_date", to_date(col("event_timestamp"))
)
    
valid_silver_df = silver_transformed.filter(
    col("transaction_id").isNotNull() &
    col("provider").isNotNull() &
    col("event_timestamp").isNotNull() &
    (col("original_amount")>=0) &
    (col("fx_rate")>=0) &
    col("provider").isin("razorpay", "slice", "dodo_payments")
)

invalid_silver_df = silver_transformed.filter(
    ~(
        col("transaction_id").isNotNull() &
        col("provider").isNotNull() &
        col("event_timestamp").isNotNull() &
        (col("original_amount")>=0) &
        (col("fx_rate")>=0) &
        col("provider").isin("razorpay", "slice", "dodo_payments")
    )
)
invalid_silver_df.write.mode("overwrite").format("delta").saveAsTable("payment_gateway_catalog.silver.silver_transactions_invalid")

target_table = DeltaTable.forName(spark, "payment_gateway_catalog.silver.silver_transactions")

target_table.alias("target").merge(
    valid_silver_df.alias("source"),
    "target.transaction_id = source.transaction_id"
).whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

print("Silver layer transformation completed with full provider attributes")

In [0]:
valid_table = spark.read.table("payment_gateway_catalog.silver.silver_transactions")
invalid_table = spark.read.table("payment_gateway_catalog.silver.silver_transactions_invalid")
display(valid_table)